# 3.2 · Random Forest y XGBoost con SHAP

**Tiempo estimado:** 45 min.

**Objetivos.**

1. Ajustar Random Forest y XGBoost vía `skforecast` con la misma matriz de features que el baseline lineal del notebook 01.
2. Comparar errores y diagnóstico visual.
3. Interpretar el modelo con **feature importance** y **SHAP**.
4. Reflexión: ¿en qué casos gana el ML clásico vs SARIMAX/Prophet de Sesión 2?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import shap
from skforecast.recursive import ForecasterRecursive

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

## 1 · Reusamos la matriz de features de 01

In [ ]:
caudal = ud.cargar_caudal_genil(source="CEDEX")
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio="2010-01-01", fecha_fin="2020-12-31")

df = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).loc["2011":"2020"]
df = df.asfreq("D").interpolate("linear", limit=7).dropna()


# Features de calendario + lluvia (los lags de q los maneja skforecast)
def features_exog(df):
    out = pd.DataFrame(index=df.index)
    for lag in (1, 2, 3, 7):
        out[f"p_lag{lag}"] = df["lluvia"].shift(lag)
    for w in (3, 7, 14, 30):
        out[f"p_acum{w}d"] = df["lluvia"].rolling(w).sum().shift(1)
    idx = out.index
    out["sin_an"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out["cos_an"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    return out.asfreq("D")


X_exog = features_exog(df).dropna().asfreq("D")
y = df["caudal"].loc[X_exog.index].asfreq("D")

split = pd.Timestamp("2018-01-01")
y_train, y_test = y.loc[:split].iloc[:-1].asfreq("D"), y.loc[split:].asfreq("D")
X_train, X_test = X_exog.loc[y_train.index].asfreq("D"), X_exog.loc[y_test.index].asfreq("D")
print(f"Train: {len(y_train):,}   Test: {len(y_test):,}   Exog: {X_train.shape[1]}")

## 2 · Ajustar 3 modelos vía skforecast

Mismos lags, distinto regresor. Predicción **recursiva** del test.

In [ ]:
LAGS = [1, 2, 3, 7, 14, 30, 90, 365]

modelos = {
    "Ridge": Ridge(alpha=1.0),
    "RF": RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6, n_jobs=-1, verbosity=0, random_state=0
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=31, n_jobs=-1, random_state=0, verbose=-1
    ),
}

preds = {}
for nombre, regr in modelos.items():
    f = ForecasterRecursive(regressor=regr, lags=LAGS)
    f.fit(y=y_train, exog=X_train)
    preds[nombre] = f.predict(steps=len(y_test), exog=X_test)
    print(f"{nombre:8s} entrenado")

In [ ]:
def rmse(y, yhat):
    return float(np.sqrt(((y - yhat) ** 2).mean()))


def mae(y, yhat):
    return float((y - yhat).abs().mean())


tabla = pd.DataFrame(
    {nombre: {"RMSE": rmse(y_test, p), "MAE": mae(y_test, p)} for nombre, p in preds.items()}
).T.round(3)
print(tabla)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, color="black", lw=1, label="observado")
colores = {"Ridge": "#999", "RF": "#2563eb", "XGBoost": "#c2410c", "LightGBM": "#16a34a"}
for n, p in preds.items():
    ax.plot(p.index, p.values, color=colores[n], lw=0.9, ls="--", label=n, alpha=0.9)
ax.set_ylabel("Q (m³/s)")
ax.legend(ncol=5)
ax.set_title("Predicción recursiva test 2018-2020")
plt.tight_layout()

## 3 · Feature importance — LightGBM

Para entender qué "mira" el modelo. Importante: en skforecast, las features incluyen los lags + las exógenas.

In [ ]:
f_lgb = ForecasterRecursive(
    regressor=lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=31, n_jobs=-1, verbose=-1, random_state=0
    ),
    lags=LAGS,
)
f_lgb.fit(y=y_train, exog=X_train)

importancia = f_lgb.get_feature_importances()
importancia.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
imp = importancia.head(15).sort_values("importance")
ax.barh(imp["feature"], imp["importance"], color="#16a34a")
ax.set_xlabel("Importance (LightGBM gain)")
plt.tight_layout()

## 4 · SHAP — interpretabilidad por predicción

Vamos al modelo subyacente, le pasamos una muestra del train, y obtenemos los Shapley values.

In [ ]:
# Construir el dataset que ve el modelo (lags + exog) — skforecast nos lo expone
X_design, y_design = f_lgb.create_train_X_y(y=y_train, exog=X_train)
print(f"Diseño: X={X_design.shape}, y={y_design.shape}")
print(X_design.head(2).round(3))

In [ ]:
# Submuestra para acelerar SHAP
rng = np.random.default_rng(0)
idx = rng.choice(len(X_design), size=min(1000, len(X_design)), replace=False)
X_sample = X_design.iloc[idx]

explainer = shap.TreeExplainer(f_lgb.regressor)
shap_values = explainer(X_sample)

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.tight_layout()

**Lectura:**

- Cada punto es un día. Color = valor del feature (rojo alto, azul bajo). Eje X = contribución a la predicción.
- `lag_1` domina: alto → alto, bajo → bajo (línea diagonal).
- `p_acum7d` y `p_acum14d`: lluvia acumulada reciente alta empuja la predicción hacia arriba.
- `sin_an` / `cos_an`: contribuyen al ciclo anual.

## 5 · Caso individual — el día de pico del test

¿Qué hizo decidir al modelo en el día con mayor caudal observado del test?

In [ ]:
# Recreamos la matriz X para el TEST
X_test_design, _ = f_lgb.create_train_X_y(
    y=pd.concat([y_train.iloc[-max(LAGS) :], y_test]),
    exog=pd.concat([X_train.iloc[-max(LAGS) :], X_test]),
)
X_test_design = X_test_design.loc[y_test.index]

fecha_pico_test = y_test.idxmax()
print(f"Pico de test: {fecha_pico_test.date()} → Q = {y_test.max():.2f} m³/s")

fila = X_test_design.loc[[fecha_pico_test]]
sv = explainer(fila)
shap.plots.waterfall(sv[0], max_display=12, show=False)
plt.tight_layout()

## 6 · Reflexión: ML vs estadísticos clásicos

Frente a SARIMAX/Prophet (Sesión 2), los modelos de árbol:

- **Ganan** cuando hay interacciones no lineales (lluvia*estación, p. ej.) y muchos features.
- **Empatan** cuando la serie es estacionaria con poca exógena (el AR clásico ya es óptimo).
- **Pierden** en extrapolación: si un pico real supera el máximo del train, no lo alcanzan.
- **Pierden** en intervalos: SARIMAX/ETS dan intervalos paramétricos; aquí necesitamos *conformal* o quantile regression.

## 7 · Ejercicios

1. **Tuning.** Usa `optuna` o `RandomizedSearchCV` (con `TimeSeriesSplit`) para tunear LightGBM. Sólo cuidado con la fuga.
2. **Quantile regression.** Cambia `objective='quantile'` y `alpha=0.9` para predecir el cuantil 90%. ¿Captura mejor los picos?
3. **Sin lluvia.** Reentrena LightGBM sin features de lluvia. ¿Cuánto pierde?
4. **Comparar modelos en NSE.** Implementa la métrica NSE del notebook 2.4 y rankea los modelos.
5. **Reto.** Concatena predicciones de LightGBM con las de SARIMAX/Prophet de Sesión 2 vía media simple. ¿Mejora?